In [0]:
# Standard library imports
import ast
import json

# Data handling and visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    classification_report,
    confusion_matrix,
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Gradient boosting
from xgboost import XGBClassifier


seed = 24601

google_analytics = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_google_analytics_abandonment.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

sales = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_sales.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

materials = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_material.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

customer = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_customer.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

cutoff_times = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_cutoff_times.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

operating_hours = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_operating_hours.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

orders = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_orders.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

visit_plan = (
    spark.read.csv(
        "/Volumes/workspace/default/capstone_data/clean_visit_plan.csv",
        header=True,
        inferSchema=True,
    ).toPandas()
)

# Standardize column names to lowercase across all dataframes
for df in [
    customer,
    cutoff_times,
    google_analytics,
    materials,
    operating_hours,
    orders,
    sales,
    visit_plan,
]:
    df.columns = df.columns.str.lower()

In [0]:
display(google_analytics.head(3))
display(sales.head(3))
display(materials.head(3))
display(customer.head(3))
display(cutoff_times.head(3))
display(operating_hours.head(3))
display(orders.head(3))
display(visit_plan.head(3))

# Helper Functions

In [0]:
def evaluate_on_test(name: str, fitted_model) -> None:
    """Print classification metrics on the test set."""
    print(f"\n##### {name} — Test Performance #####")
    y_pred = fitted_model.predict(X_test)
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, digits=3))
    if hasattr(fitted_model, "predict_proba"):
        y_prob = fitted_model.predict_proba(X_test)[:, 1]
        ap = average_precision_score(y_test, y_prob)
        print(f"Average Precision (PR-AUC): {ap:.4f}")


def downsample(
    Xin: pd.DataFrame,
    yin: pd.Series,
    ratio: int = 2
) -> tuple[pd.DataFrame, pd.Series]:
    """Downsample majority class to a given ratio."""
    vc = yin.value_counts()
    maj = vc.idxmax()
    mino = vc.idxmin()
    n_min = vc[mino]
    n_maj_tgt = ratio * n_min

    rng = np.random.RandomState(seed)
    maj_idx = yin[yin == maj].index
    keep_maj = rng.choice(
        maj_idx, size=min(n_maj_tgt, len(maj_idx)), replace=False
    )

    keep_idx = np.concatenate([yin[yin == mino].index, keep_maj])
    X_out = Xin.loc[keep_idx].reset_index(drop=True)
    y_out = yin.loc[keep_idx].reset_index(drop=True)
    return X_out, y_out


def build_preprocessor(
    Xtrain: pd.DataFrame
) -> tuple[ColumnTransformer, list[str], list[str]]:
    """Build a numeric+categorical preprocessing pipeline."""
    num_cols = Xtrain.select_dtypes(
        include=["int64", "float64"]
    ).columns.tolist()
    cat_cols = Xtrain.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    pre = ColumnTransformer(
        [
            ("num", StandardScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ]
    )
    return pre, num_cols, cat_cols


def pred_proba_best(bst, dmat) -> np.ndarray:
    """Predict probabilities using the best iteration/tree limit when set."""
    if hasattr(bst, "best_iteration") and bst.best_iteration is not None:
        try:
            return bst.predict(
                dmat, iteration_range=(0, bst.best_iteration + 1)
            )
        except TypeError:
            pass

    if hasattr(bst, "best_ntree_limit") and bst.best_ntree_limit is not None:
        return bst.predict(dmat, ntree_limit=bst.best_ntree_limit)

    return bst.predict(dmat)

# Preprocessing


In [0]:
# Get unique material-to-trademark combinations
unique_combos = materials[["material_id", "trademark"]].drop_duplicates()

# print(unique_combos.shape)
# print(unique_combos.head(20))

[Preprocessing]

This step above extracts a clean mapping between each material ID and its associated trademark (brand) by removing duplicate pairs. Since the raw product catalog may contain repeated entries, this ensures a 1 to 1 lookup between material ids and their brand labels. This clean mapping will be used later during feature engineering to enrich transactional and abandoned cart records with brand level information, enabling downstream modeling to analyze abandonment patterns by product brand.

In [0]:
events = google_analytics.copy()

# Remove inactive records based on specific conditions
inactive_mask = (
    (events['abandoned'] == False)
    & (events['false_by_purchase'] == 'no purchase')
    & (events['recovered'] == 'not applicable')
)

# Filter out inactive records
events = events[~inactive_mask]

# Normalize and parse the "items" column into lists of dictionaries
events['items'] = (
    events['items']
    .fillna('[]')
    .astype(str)
    .str.strip()
    .apply(
        lambda s: []
        if s in ('', '[]')
        else (json.loads(s) if '"' in s else ast.literal_eval(s))
    )
)

# Expand the "items" list so each element becomes its own row
events = events.explode('items', ignore_index=True)

# Extract "item_id" and "quantity" fields from each item dictionary
events['item_id'] = events['items'].map(
    lambda d: d.get('item_id') if isinstance(d, dict) else None
)
events['quantity'] = events['items'].map(
    lambda d: d.get('quantity') if isinstance(d, dict) else None
)

# Standardize data types for extracted columns
events['item_id'] = events['item_id'].astype('string')
events['quantity'] = pd.to_numeric(events['quantity'], errors='coerce').astype('Int64')

# Remove rows where "item_id" is missing
events = events[events['item_id'].notna()].reset_index(drop=True)

# Preview the transformed data
print(events.head())

# Drop the 'items' column now that its contents have been expanded
events = events.drop(columns=['items'])

# Inspect unique item IDs
events['item_id'].unique()

# Remove records with non-numeric item IDs starting with '01t'
events = events[~events['item_id'].str.startswith('01t')].reset_index(drop=True)

# Convert item_id column to integer type
events['item_id'] = events['item_id'].astype(int)

# Review class balance for the 'abandoned' target variable
events['abandoned'].value_counts(normalize=True)

# Note imbalance for downsampling consideration
print(
    'The majority class (abandoned = False) represents approximately 90%, '
    'so we will downsample the True class to achieve a 2:1 ratio.'
)

# Check for missing values before modeling
display(events.isnull().sum())

# Remove records with null values
post_eventns = events.dropna().reset_index(drop=True)

# # Display raw counts of the target variable
# display(post_eventns['abandoned'].value_counts())

# # Display normalized class proportions
# display(post_eventns['abandoned'].value_counts(normalize=True))

[Preprocessing]

This code above prepares raw Google Analytics event data for modeling by filtering out clearly inactive or irrelevant records and expanding the nested items field into a row per product structure. It goes through the serialized JSON lists into usable dictionaries, then extracts item_id and quantity attributes to enable product level analysis. Invalid entries are deleted, and item_id values are standardized to numeric formats, with non product identifiers discarded. We also inspect and quantify the imbalance in the abandoned target, which later informs us about the downsampling strategy to use. Finally, all remaining missing values are removed to ensure data integrity before training as Null data will prevent the model from running. This structured event level dataset allows downstream models to learn relationships between product behavior, device context, and abandonment outcomes.

In [0]:
# Copy and prepare dataframe
df_prep = post_eventns.copy()

# Parse timestamp columns
df_prep["event_date"] = pd.to_datetime(df_prep["event_date"], errors="coerce")
df_prep["event_ts_utc"] = pd.to_datetime(df_prep["event_ts_utc"], errors="coerce")

# Create time-based features
df_prep["dow"] = df_prep["event_date"].dt.dayofweek
df_prep["hour"] = df_prep["event_ts_utc"].dt.hour

# Drop nulls after feature creation
df_prep = df_prep.dropna().reset_index(drop=True)
# print("After dropna:", df_prep.shape)

# Define features and target
y = df_prep["abandoned"].astype(int)
feature_cols = [
    "event_name",
    "device_category",
    "device_mobile_brand_name",
    "device_operating_system",
    "event_page_name",
    "quantity",
    "dow",
    "hour",
]
X = df_prep[feature_cols]

# Split train and test data
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=seed
)

# Downsample majority class for balance
counts = y_train_full.value_counts()
maj_label = counts.idxmax()
min_label = counts.idxmin()
n_min = counts[min_label]
n_maj_target = 2 * n_min

maj_idx = y_train_full[y_train_full == maj_label].index
min_idx = y_train_full[y_train_full == min_label].index

rng = np.random.RandomState(seed)
maj_idx_down = rng.choice(maj_idx, size=min(n_maj_target, len(maj_idx)), replace=False)
down_idx = np.concatenate([min_idx, maj_idx_down])

X_train = X_train_full.loc[down_idx].reset_index(drop=True)
y_train = y_train_full.loc[down_idx].reset_index(drop=True)

print("Train class balance BEFORE:", counts.to_dict())
print("Train class balance AFTER: ", y_train.value_counts().to_dict())

# Define preprocessing pipeline
numeric_cols = X_train.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

# Create CV subset for tuning
cv_frac = 0.6
X_train_cv, _, y_train_cv, _ = train_test_split(
    X_train, y_train, train_size=cv_frac, stratify=y_train, random_state=seed
)
print(f"Using {cv_frac*100:.0f}% of downsampled train for CV:", X_train_cv.shape)

[Preprocessing]

This block prepares the cleaned event level dataset for machine learning by engineering time based features, handling class imbalance, and constructing a preprocessing pipeline. First, timestamp columns are parsed into proper datetime formats and transformed into more model friendly attributes such as day of week and hour of activity, which can help capture behavioral patterns tied to cart abandonment timing. After removing null values, we define the target variable and select a set of device, event, and product related predictor features.

The data is then split into training and testing sets using stratification to preserve the original class proportions. Stratification is so essential for this as it causes the train and test sets to be simillar. Because abandonment is rare, we downsample the majority class to achieve a more balanced 2:1 ratio, improving the model’s ability to learn minority class patterns. We also identify numerical and categorical columns, scaling numeric values and one hot encoding categories using ColumnTransformer. Finally, we create a cross validation subset from the downsampled training data to speed up hyperparameter tuning. Overall these preprocessing steps ensure that the training data is balanced, standardized, and ready for modeling.

# Modeling

In [0]:
# Initialize random forest model
rf = RandomForestClassifier(
    random_state=seed,
    n_jobs=-1,
    max_samples=0.7,
    bootstrap=True
)

# Create model pipeline
rf_pipe = Pipeline([("prep", preprocessor), ("model", rf)])

# Define hyperparameter grid
rf_params = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [8, None],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", "log2"],
}

# Run randomized search CV
rf_search = RandomizedSearchCV(
    rf_pipe,
    rf_params,
    n_iter=3,
    cv=2,
    scoring="f1",
    n_jobs=-1,
    random_state=seed,
    verbose=2,
)

print("\nRF: quick CV on subset")
rf_search.fit(X_train_cv, y_train_cv)
print("RF best params (subset):", rf_search.best_params_)

# Refit best model on full training data
rf_best = rf_search.best_estimator_
rf_best.fit(X_train, y_train)

# Evaluate performance on test data
evaluate_on_test("Random Forest", rf_best)

[Interpretation]

We trained and tuned a Random Forest model to predict cart abandonment using a small randomized hyperparameter search, and the best configuration selected 150 trees with unrestricted depth, a slightly more conservative minimum split size, and log2 feature sampling to increase diversity among trees. This combination suggests the model benefits from flexibility to capture non linear patterns while still controlling overfitting. Random forest is excellent at modelling non linear data. <br> 
Evaluated on the test set, the model achieved an overall accuracy of 82.8%, which appears strong at first glance. However, because abandonment is a rare event in our dataset, accuracy alone is misleading. The confusion matrix reveals that the model performs very well at identifying customers who do not abandon (True Negative rate of 93.1 percent), but struggles to correctly detect customers who actually abandoned. For the minority class, precision was 27.3 percent and recall was 16.7 percent, resulting in an F1-score of 0.207. In other words, when the model predicts abandonment, it is only correct about one quarter of the time, and it currently only catches about one sixth of true abandoners. The low recall indicates many lost revenue opportunities are still slipping through undetected.

The macro average performance metrics (F1 = 0.555) more honestly reflect the imbalance in the data, while the weighted-average metrics are skewed by the dominant class. We also calculated the Precision-Recall AUC (0.205), which is an appropriate metric for rare event modeling. This value indicates the model performs moderately above random, but there is substantial room for improvement. Overall, this version of the Random Forest is useful as a ranking tool, as it can help us prioritize which customers are more likely to abandon relative to others, but it is not yet strong enough to serve as a direct trigger for retention interventions.

In [0]:
# Initialize the logistic regression model
log_reg = LogisticRegression(
    solver='saga',
    penalty='l2',
    C=1.0,
    class_weight='balanced',
    max_iter=500,
    tol=1e-3,
    warm_start=False,
    random_state=seed,
)

# Create a modeling pipeline without caching
log_pipe = Pipeline(
    steps=[
        ('prep', preprocessor),
        ('model', log_reg),
    ]
)

# Define a simple hyperparameter grid for regularization strength
log_params = {'model__C': [0.1, 0.3, 1.0]}

# Perform a randomized cross-validation search
log_search = RandomizedSearchCV(
    estimator=log_pipe,
    param_distributions=log_params,
    n_iter=3,
    cv=2,
    scoring='f1',
    n_jobs=1,
    random_state=seed,
    verbose=2,
    refit=True,
)

# Fit the randomized search on a CV subset
print('\nLogReg: quick CV on subset')
log_search.fit(X_train_cv, y_train_cv)
print('LogReg best params (subset):', log_search.best_params_)

# Rebuild the best model using the optimal hyperparameters
best_C = log_search.best_params_['model__C']
log_best = Pipeline(
    steps=[
        ('prep', preprocessor),
        (
            'model',
            LogisticRegression(
                solver='saga',
                penalty='l2',
                C=best_C,
                class_weight='balanced',
                max_iter=1000,
                tol=1e-3,
                warm_start=False,
                random_state=seed,
            ),
        ),
    ]
)

# Fit the final model on the full training dataset
print('\nRefitting best Logistic Regression on FULL train')
log_best.fit(X_train, y_train)

# Evaluate model performance on the test dataset
evaluate_on_test('Logistic Regression', log_best)

[Interpretation]

We trained and tuned a Logistic Regression model using a small randomized hyperparameter search focused on regularization strength. The best performing model used a relatively stronger regularization value (C = 0.1). Lower values of C apply more regularization, which helps prevent overfitting and encourages the model to rely on only the most consistently predictive signals. Because Logistic Regression is inherently linear, it tends to prefer simpler solutions, and the class_weight='balanced' parameter was included to compensate for the severe imbalance between abandoned and non abandoned cases.

When evaluated on the test set, this model achieved an overall accuracy of 52.8 percent. Although accuracy appears lower than tree based models, this drop is expected because Logistic Regression does not automatically learn complex non linear interactions. The confusion matrix shows that the model correctly identifies some abandoners that the Random Forest missed, but it also incorrectly flags a large number of customers who did not abandon. This trade off occurs because class balancing pushes the model to be more aggressive in predicting the minority class. As a result, precision and recall for the abandoned class are more balanced, but still modest, and the F1-score indicates that performance remains limited in this challenging rare-event setting.

From a fairness and ranking perspective, Logistic Regression offers clear coefficient-based interpretability, allowing us to understand which features most strongly increase or decrease abandonment risk. However, its linear decision boundary limits its ability to capture complex patterns in customer behavior and cart activity. The macro average metrics again illustrate the impact of class imbalance, while the weighted averages are skewed toward the dominant non abandonment class. The Precision Recall AUC score of 0.194 suggests slightly above random performance, but it underperforms the Random Forest’s ability to isolate subtle abandonment behavior.

Overall, Logistic Regression provides a useful baseline model that is fast, interpretable, and simple, but it is not well suited to modeling the complex, non linear signals involved in cart abandonment.

In [0]:
# Initialize XGBoost model
xgb = XGBClassifier(
    random_state=seed,
    tree_method="hist",
    eval_metric="logloss",
    n_jobs=-1
)

# Create model pipeline
xgb_pipe = Pipeline([("prep", preprocessor), ("model", xgb)])

# Define hyperparameter grid
xgb_params = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [4, 6],
    "model__learning_rate": [0.05, 0.1],
    "model__subsample": [0.8, 1.0],
    "model__colsample_bytree": [0.8, 1.0],
}

# Run randomized search CV
xgb_search = RandomizedSearchCV(
    xgb_pipe,
    xgb_params,
    n_iter=3,
    cv=2,
    scoring="f1",
    n_jobs=-1,
    random_state=seed,
    verbose=2,
)

print("\nXGB: quick CV on subset")
xgb_search.fit(X_train_cv, y_train_cv)
print("XGB best params (subset):", xgb_search.best_params_)

# Refit best model on full training data
xgb_best = xgb_search.best_estimator_
xgb_best.fit(X_train, y_train)

# Evaluate performance on test data
evaluate_on_test("XGBoost", xgb_best)

[Interpretation]

We trained and tuned an XGBoost classification model using a small randomized hyperparameter search that explored the number of boosting rounds, tree depth, learning rate, and sampling strategies. The best performing configuration used 250 trees, a relatively shallow max depth of 4, and a learning rate of 0.1. This setup suggests that the model benefits from taking a larger number of smaller, incremental steps while keeping individual trees simple. The subsample rate of 0.8 and full column sampling (colsample_bytree = 1.0) help introduce controlled randomness, which reduces overfitting and increases generalization.

When evaluated on the test set, XGBoost achieved an overall accuracy of 83.0 percent. Similar to the Random Forest, this high accuracy is mostly driven by strong performance on the dominant non abandonment carts. The confusion matrix shows a True Negative rate of 93.4 percent, indicating the model is very reliable at identifying customers who will not abandon. However, recall for the abandoned class remains relatively low at 16.0 percent, meaning the model is only capturing a small portion of customers who actually abandon their carts. Precision for this class is also modest at 27.4 percent, so only about one quarter of predicted abandoners truly abandon. Together, these scores produce a minority class F1-score of 0.202, which reflects the difficulty of detecting rare abandonment behavior.

The macro averaged metrics (F1 = 0.554) better represent the imbalance in the outcome variable, whereas the weighted average is skewed by the large volume of non abandonment. The Precision Recall AUC score of 0.2042 suggests performance moderately above random, and it nearly matches the Random Forest, indicating that both tree-based approaches are capturing similar signal from our features.

Overall, XGBoost provides competitive performance, efficiently models non linear relationships, and benefits from boosting based refinement that sequentially corrects prior errors. However, like the other models, it still struggles to identify abandoners due to severe class imbalance and limited signal in the event data.

In [0]:
# Prepare dataframe and build features
post_eventns_x = post_eventns.copy()
post_eventns_x["event_date"] = pd.to_datetime(
    post_eventns_x["event_date"], errors="coerce"
)
post_eventns_x["event_ts_utc"] = pd.to_datetime(
    post_eventns_x["event_ts_utc"], errors="coerce"
)
post_eventns_x["dow"] = post_eventns_x["event_date"].dt.dayofweek
post_eventns_x["hour"] = post_eventns_x["event_ts_utc"].dt.hour
post_eventns_x = post_eventns_x.dropna().reset_index(drop=True)

# Define features and target
y = post_eventns_x["abandoned"].astype(int)
feat_cols = [
    "event_name",
    "device_category",
    "device_mobile_brand_name",
    "device_operating_system",
    "event_page_name",
    "quantity",
    "dow",
    "hour",
]
X = post_eventns_x[feat_cols]

# Split into train and test sets
X_tr_all, X_test, y_tr_all, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=seed
)

# Create validation split from training data
X_tr_raw, X_val_raw, y_tr_raw, y_val_raw = train_test_split(
    X_tr_all, y_tr_all, test_size=0.10, stratify=y_tr_all, random_state=seed
)

# Build downsampled train variants
train_variants = {
    "ds_2to1": downsample(X_tr_raw, y_tr_raw, ratio=2),
    "ds_1to1": downsample(X_tr_raw, y_tr_raw, ratio=1),
}

# Run model tuning and evaluation with sklearn XGBClassifier
configs = []
param_grid = [
    {
        "eta": 0.05,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "spw": 1.0,
    },
    {
        "eta": 0.05,
        "max_depth": 8,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "spw": 1.2,
    },
    {
        "eta": 0.10,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "spw": 1.0,
    },
    {
        "eta": 0.10,
        "max_depth": 6,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "spw": 1.5,
    },
    {
        "eta": 0.05,
        "max_depth": 6,
        "subsample": 0.9,
        "colsample_bytree": 0.9,
        "spw": 1.5,
    },
    {
        "eta": 0.10,
        "max_depth": 4,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "spw": 2.0,
    },
]

# Train and evaluate across parameter grid
for tag, (Xtr, ytr) in train_variants.items():
    pre, num_cols, cat_cols = build_preprocessor(Xtr)
    Xtr_p = pre.fit_transform(Xtr)
    Xval_p = pre.transform(X_val_raw)
    Xtest_p = pre.transform(X_test)

    neg = int((ytr == 0).sum())
    pos = int((ytr == 1).sum())
    base_spw = max(1.0, neg / max(1, pos))

    for p in param_grid:
        clf = XGBClassifier(
            objective="binary:logistic",
            tree_method="hist",
            eval_metric="logloss",
            n_estimators=800,
            learning_rate=p["eta"],
            max_depth=p["max_depth"],
            subsample=p["subsample"],
            colsample_bytree=p["colsample_bytree"],
            scale_pos_weight=base_spw * p["spw"],
            random_state=seed,
            n_jobs=-1,
        )

        # Fit the model using validation set; omit early_stopping_rounds for compatibility
        clf.fit(
            Xtr_p,
            ytr,
            eval_set=[(Xval_p, y_val_raw)],
            verbose=False,
        )

        best_iter = getattr(clf, "best_iteration_", None)

        yv = clf.predict_proba(Xval_p)[:, 1]
        prec, rec, thr = precision_recall_curve(y_val_raw, yv)
        f1 = 2 * (prec * rec) / (prec + rec + 1e-12)
        best_idx = int(np.nanargmax(f1))
        thr_star = thr[min(best_idx, len(thr) - 1)] if len(thr) > 0 else 0.5
        val_f1 = float(f1[best_idx])
        val_ap = float(average_precision_score(y_val_raw, yv))

        configs.append(
            {
                "variant": tag,
                "params": p,
                "best_iter": best_iter,
                "val_f1": val_f1,
                "val_ap": val_ap,
                "thr": float(thr_star),
                "clf": clf,
                "pre": pre,
                "Xtest_p": Xtest_p,
            }
        )

# Rank and display leaderboard
configs = sorted(configs, key=lambda z: z["val_f1"], reverse=True)
print("\nXGB Leaderboard (by Validation F1 for class=1)")
for r in configs[:5]:
    print(
        f"{r['variant']} | F1={r['val_f1']:.3f} | PR-AUC={r['val_ap']:.3f} "
        f"| thr={r['thr']:.3f} | iter={r['best_iter']} | params={r['params']}"
    )

# Evaluate best model on test data
best = configs[0]
pre = best["pre"]
Xtest_p = best["Xtest_p"]
y_test_prob = best["clf"].predict_proba(Xtest_p)[:, 1]
y_test_pred = (y_test_prob >= best["thr"]).astype(int)

print("\nTEST at tuned threshold (picked on VAL F1)")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
print(classification_report(y_test, y_test_pred, digits=3))
print(f"Test PR-AUC: {average_precision_score(y_test, y_test_prob):.4f}")
print("\nBest config used:")
print(
    best["variant"],
    best["params"],
    "best_iter:",
    best["best_iter"],
    "thr:",
    round(best["thr"], 3),
)

[Interpretation]

In this test, we trained multiple XGBoost models using different hyperparameter configurations and applied downsampling to partially rebalance the dataset before fitting. Because abandonment is rare, we also tuned the scale_pos_weight parameter, which increases the loss penalty when the model misclassifies an abandoned cart. We evaluated all configurations on a held out validation set and selected the model that maximized the F1 score for the abandoned class. Unlike earlier experiments, we then set the final decision threshold using the point on the validation curve that provided the best precision recall trade-off for abandoners, rather than relying on the default 0.50 cutoff.

On the test set, this tuned XGBoost model demonstrated much higher recall for abandoned carts (68.2 percent), meaning it successfully identifies a much larger portion of customers who will ultimately abandon. However, this improvement comes at the cost of substantially lower precision (17.9 percent), indicating many false positives. In practice, this means the model is aggressive. it catches far more true abandoners but also incorrectly flags many customers who will end up purchasing. Overall accuracy drops to 53.6 percent, which is expected because we intentionally shifted the threshold to favor minority-class recall.

The Test PR-AUC of 0.2053 closely matches our earlier boosting results, suggesting that even with threshold tuning and downsampling, abandonment remains difficult to predict from event level features alone. This configuration is useful if the business goal is to maximize saves and is willing to tolerate more outreach to non-abandoners. However, additional behavioral signals, cart context, or historical customer profiles would likely be required to raise both precision and recall simultaneously.

In [0]:
# Calculate abandonment rate by device category
device_abandon = (
    post_eventns.groupby("device_category")["abandoned"]
    .mean()
    .reset_index()
    .rename(columns={"abandoned": "abandonment_rate"})
    .sort_values("abandonment_rate", ascending=False)
)

plt.figure(figsize=(7, 4))
sns.barplot(
    x="device_category",
    y="abandonment_rate",
    data=device_abandon,
    palette="crest"
)
plt.title("Cart Abandonment Rate by Device Type")
plt.ylabel("Abandonment Rate (%)")
plt.show()

[Interpretation]

Out of all three device types, tablets have the highest abandonment rate followed by mobile then desktop.

In [0]:
# Filter data to only abandoned records
abandoned_post_eventns = post_eventns[post_eventns["abandoned"] == True].copy()

# Count abandoned item IDs
item_counts = (
    abandoned_post_eventns["item_id"]
    .astype(str)
    .value_counts()
    .to_frame("count")
    .reset_index()
    .rename(columns={"index": "item_id"})
)

# Identify product name column in materials
name_col = None
for candidate in ["trademark", "trade_mark_desc", "product_name", "material_desc", "name"]:
    if candidate in materials.columns:
        name_col = candidate
        break

if name_col is None:
    raise KeyError(
        "Couldn't find a product name/description column in `materials`. "
        "Tried: ['trademark','trade_mark_desc','product_name','material_desc','name'].\n"
        f"materials columns are: {list(materials.columns)}"
    )

# Ensure merge key data type alignment
materials = materials.copy()
if "material_id" not in materials.columns:
    raise KeyError("`materials` is missing 'material_id' column.")
materials["material_id"] = materials["material_id"].astype(str)

# Merge item counts with materials data
merged = item_counts.merge(
    materials[["material_id", name_col]],
    left_on="item_id",
    right_on="material_id",
    how="left"
)

# Combine counts by product or trademark
grouped = (
    merged.groupby(name_col, dropna=False)["count"]
    .sum()
    .reset_index()
    .rename(columns={name_col: "product_name"})
    .sort_values("count", ascending=False)
)

# Select top 10 abandoned products
top10_products = grouped.head(10).reset_index(drop=True)

# Plot top abandoned products
plt.figure(figsize=(9, 5))
labels = top10_products["product_name"].fillna("(unknown)")
counts = top10_products["count"]

plt.barh(labels[::-1], counts[::-1])
plt.xlabel("Number of Abandoned Carts")
plt.ylabel("Product (Trademark)")
plt.title("Top 10 Most Frequently Abandoned Products")
plt.tight_layout()
plt.show()

[Interpretation]

The top 3 most frequently abandoned products are: Fizz Factory, Oliver Originals and Petes Popcorn.

# Results

Q3 Results - What is the quantitative answer to the question? 

Out of all device types, Tablets have the highest abandoned rates followed by mobile, then desktops. With an abandoned rate of 21.8% for tablets, this could be due to UI difficulties. This is significant as tablets themselves are the rarest device used when navigating through MyCoke360, yet represent the highest abandon rate. Mobile is close to the abandonment rate as Tablet at 20.7%. 

Q4 Results - What is the quantitative answer to the question?

Based on our aggregated analysis of all abandoned cart events, Fizz Factory is the most frequently abandoned brand on the platform, appearing in approximately 250,000 abandoned carts. This represents a significantly larger share of abandonment volume compared to other brands, suggesting either high browsing interest without conversion or potential friction related to those products. The second most abandoned brand is Oliver Originals, with roughly 145,000 abandoned carts, followed by Pete’s Popcorn at approximately 60,000. 

Q3 Recommendation(s)
The abandonment rate in tablets and mobile could indiciate difficulties due to ordering with a smaller screen size. Revising the MyCoke360 webpage to ease Tablet and Mobile orders could be worth implementing and the results can be later compared via an A/B test.

Q4 Recommendation(s)
The sharp drop off after the first two brands (Fizz Factory and Oliver Originals) indicates a distribution where only a small number of brands account for the majority of abandonment. These insights can help prioritize retention strategies, such as targeted promotions, improved product page content, or restocking visibility for the most affected brands which SWIRE can implement.

# Contributions - Ali
- Created a Random Forest, Logistic Regression, XGBoost Model + tuned version. 
- Added interpretations to modelling and some charts. 
- Answered Key Questions: 
    1. Abandonment Rate by Device 
    2. Top 10 abandoned products.
